# 🎓 SyllaBot — Checkpoint 1: Text Preprocessing Pipeline
**IS Professional Elective #4 — Generative AI Systems**  
**Group: Sugar Group | A.Y. 2026–2027**

This notebook covers **Deliverable #3** of Checkpoint 1:
- Load PDF and DOCX syllabus files
- Clean and normalize raw text
- Tokenize the processed text
- Show before/after examples


## Step 1 — Install Required Libraries

In [ ]:
# Install all required packages
!pip install pdfplumber python-docx nltk -q

## Step 2 — Upload Your Syllabus Files
Run this cell to upload your PDF and DOCX files from your computer.

## Step 2.1 — Pull Syllabi from GitHub (Optional)

If your syllabus files are hosted on GitHub, run this cell to clone your repository and pull them into the `syllabi` folder. This is an alternative to Step 2 (manual upload).

In [ ]:
import os

# --- Configuration for GitHub Pull ---
# 1. Replace with your actual GitHub repository URL
github_repo_url = 'https://github.com/Toshikiru/SyllaBot-A-RAG-Based-Course-Syllabus-and-Study-Guide-Assistant.git'

# 2. Point to the folder inside the cloned repo that contains your PDFs/DOCXs.
#    If your files are in the root of the repo, just use the repository's main folder name (e.g., 'YourRepoName').
github_folder = 'Checkpoint1/data_source_raw' # Example: 'MyRepo/data_files'

# --- Clone and Move Files ---
print(f'Attempting to clone {github_repo_url.split("/")[-1].replace(".git", "")}...')
repo_name = github_repo_url.split('/')[-1].replace('.git', '')
!git clone {github_repo_url}

# Create the target 'syllabi' folder if it doesn't exist (already handled by previous cell, but good for robustness)
os.makedirs('syllabi', exist_ok=True)

source_path = os.path.join(repo_name, github_folder)

if os.path.exists(source_path):
    # Find all PDF and DOCX files in the specified GitHub folder
    files_to_move = [f for f in os.listdir(source_path) if f.lower().endswith(('.pdf', '.docx'))]

    for filename in files_to_move:
        src = os.path.join(source_path, filename)
        dst = os.path.join('syllabi', filename) # Move to the 'syllabi' folder

        os.rename(src, dst)
        print(f'  ✅ Pulled from GitHub and Saved to syllabi/: {filename}')

    print(f'\n📁 Total files pulled from GitHub: {len(files_to_move)}')
else:
    print(f'❌ Source folder "{source_path}" not found after cloning. Check your `github_folder` path!')

# Clean up the cloned repository folder to avoid clutter
!rm -rf {repo_name}
print(f'Cleaned up temporary repository folder: {repo_name}')

Attempting to clone SyllaBot-A-RAG-Based-Course-Syllabus-and-Study-Guide-Assistant...
Cloning into 'SyllaBot-A-RAG-Based-Course-Syllabus-and-Study-Guide-Assistant'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 24 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 1.66 MiB | 26.15 MiB/s, done.
Resolving deltas: 100% (3/3), done.
  ✅ Pulled from GitHub and Saved to syllabi/: 01_GenAI_Systems_Syllabus.docx
  ✅ Pulled from GitHub and Saved to syllabi/: 06_Academic_Calendar_and_Policies.pdf
  ✅ Pulled from GitHub and Saved to syllabi/: 07_IT_Ethics_Syllabus.pdf
  ✅ Pulled from GitHub and Saved to syllabi/: 02_Business_Analytics_Syllabus.docx
  ✅ Pulled from GitHub and Saved to syllabi/: 03_Information_Management_Syllabus.pdf
  ✅ Pulled from GitHub and Saved to syllabi/: 04_Systems_Analysis_Design_Syllabus.pdf
  ✅ Pulled from GitHub and Saved to syllabi

## Step 3 — Extract Raw Text from Files
We extract raw text from both PDF and DOCX formats.

In [ ]:
import pdfplumber
from docx import Document

def extract_text_from_pdf(filepath):
    """Extract raw text from a PDF file."""
    text = ''
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + '\n'
    return text

def extract_text_from_docx(filepath):
    """Extract raw text from a DOCX file."""
    doc = Document(filepath)
    text = '\n'.join([para.text for para in doc.paragraphs if para.text.strip()])
    return text

# Load all files from the syllabi folder
raw_documents = {}  # {filename: raw_text}

for filename in os.listdir('syllabi'):
    filepath = f'syllabi/{filename}'
    if filename.endswith('.pdf'):
        raw_documents[filename] = extract_text_from_pdf(filepath)
        print(f'  📄 Extracted PDF: {filename} ({len(raw_documents[filename])} chars)')
    elif filename.endswith('.docx'):
        raw_documents[filename] = extract_text_from_docx(filepath)
        print(f'  📝 Extracted DOCX: {filename} ({len(raw_documents[filename])} chars)')
    else:
        print(f'  ⚠️  Skipped unsupported file: {filename}')

print(f'\n✅ Total documents loaded: {len(raw_documents)}')

  📝 Extracted DOCX: 01_GenAI_Systems_Syllabus.docx (2656 chars)
  📄 Extracted PDF: 06_Academic_Calendar_and_Policies.pdf (3825 chars)
  📄 Extracted PDF: 07_IT_Ethics_Syllabus.pdf (3583 chars)
  📝 Extracted DOCX: 02_Business_Analytics_Syllabus.docx (2071 chars)
  📄 Extracted PDF: 03_Information_Management_Syllabus.pdf (2710 chars)
  📄 Extracted PDF: 04_Systems_Analysis_Design_Syllabus.pdf (2735 chars)
  📄 Extracted PDF: 08_Software_Engineering_Syllabus.pdf (3425 chars)
  📄 Extracted PDF: 05_Thesis_Writing_Syllabus.pdf (3188 chars)

✅ Total documents loaded: 8


## Step 4 — BEFORE Example (Raw Text Preview)
This shows what the text looks like **before** cleaning.

In [ ]:
# Pick the first document to show the before/after example
sample_filename = list(raw_documents.keys())[0]
sample_raw = raw_documents[sample_filename]

print('=' * 60)
print(f'📄 BEFORE CLEANING — {sample_filename}')
print('=' * 60)
print(sample_raw[:1000])  # Show first 1000 characters
print('...')

📄 BEFORE CLEANING — 01_GenAI_Systems_Syllabus.docx
Course Syllabus
IS Professional Elective #4 — Generative AI Systems
Instructor: Jessie A. Melendres | 1st Semester, A.Y. 2026–2027 | Units: 3 | Schedule: TTh 1:00–2:30 PM
1. Course Description
This course introduces students to the principles and applications of Generative AI systems. Topics include transformer architecture, large language models (LLMs), prompt engineering, retrieval-augmented generation (RAG), vector databases, fine-tuning techniques such as LoRA and QLoRA, and LLMOps deployment practices. Students will design and build a working Generative AI application as a capstone mini-project that progresses through four checkpoints aligned with the four exam periods.
2. Course Intended Learning Outcomes (CILOs)
By the end of this course, students should be able to:
CILO 1: Build a full end-to-end data pipeline including text cleaning, tokenization, embedding generation, and dimensionality reduction.
CILO 2: Design and implement

## Step 5 — Text Cleaning & Normalization
We remove noise: extra whitespace, special characters, headers/footers, and empty lines.

In [ ]:
import re

def clean_text(text):
    """
    Clean and normalize raw text extracted from syllabus documents.
    Steps:
      1. Remove non-ASCII characters
      2. Remove URLs
      3. Remove extra whitespace and blank lines
      4. Normalize multiple spaces into one
      5. Strip leading/trailing whitespace per line
      6. Convert to lowercase
    """
    # 1. Remove non-ASCII characters (e.g., weird PDF symbols)
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 2. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 3. Remove special characters except basic punctuation
    text = re.sub(r'[^\w\s.,;:\-\'"()%/]', ' ', text)

    # 4. Normalize multiple spaces into one
    text = re.sub(r' +', ' ', text)

    # 5. Remove lines that are just numbers (page numbers)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 6. Strip each line and remove blank lines
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    text = '\n'.join(lines)

    # 7. Convert to lowercase
    text = text.lower()

    return text

# Apply cleaning to all documents
cleaned_documents = {}
for filename, raw_text in raw_documents.items():
    cleaned_documents[filename] = clean_text(raw_text)
    print(f'  ✅ Cleaned: {filename}')

print('\n🧹 All documents cleaned!')

  ✅ Cleaned: 01_GenAI_Systems_Syllabus.docx
  ✅ Cleaned: 06_Academic_Calendar_and_Policies.pdf
  ✅ Cleaned: 07_IT_Ethics_Syllabus.pdf
  ✅ Cleaned: 02_Business_Analytics_Syllabus.docx
  ✅ Cleaned: 03_Information_Management_Syllabus.pdf
  ✅ Cleaned: 04_Systems_Analysis_Design_Syllabus.pdf
  ✅ Cleaned: 08_Software_Engineering_Syllabus.pdf
  ✅ Cleaned: 05_Thesis_Writing_Syllabus.pdf

🧹 All documents cleaned!


## Step 6 — AFTER Example (Cleaned Text Preview)
This shows what the same text looks like **after** cleaning.

In [ ]:
sample_cleaned = cleaned_documents[sample_filename]

print('=' * 60)
print(f'✨ AFTER CLEANING — {sample_filename}')
print('=' * 60)
print(sample_cleaned[:1000])
print('...')

# Show size reduction
before_len = len(raw_documents[sample_filename])
after_len = len(sample_cleaned)
reduction = round((1 - after_len / before_len) * 100, 2)
print(f'\n📊 Size before: {before_len} chars')
print(f'📊 Size after:  {after_len} chars')
print(f'📉 Noise reduced by: {reduction}%')

✨ AFTER CLEANING — 01_GenAI_Systems_Syllabus.docx
course syllabus
is professional elective 4 generative ai systems
instructor: jessie a. melendres 1st semester, a.y. 20262027 units: 3 schedule: tth 1:002:30 pm
1. course description
this course introduces students to the principles and applications of generative ai systems. topics include transformer architecture, large language models (llms), prompt engineering, retrieval-augmented generation (rag), vector databases, fine-tuning techniques such as lora and qlora, and llmops deployment practices. students will design and build a working generative ai application as a capstone mini-project that progresses through four checkpoints aligned with the four exam periods.
2. course intended learning outcomes (cilos)
by the end of this course, students should be able to:
cilo 1: build a full end-to-end data pipeline including text cleaning, tokenization, embedding generation, and dimensionality reduction.
cilo 2: design and implement a scalable 

## Step 7 — Tokenization
We split the cleaned text into individual word tokens using NLTK.

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def tokenize_text(text):
    """
    Tokenize cleaned text into individual word tokens.
    Also removes stopwords (common words like 'the', 'is', 'and').
    """
    tokens = word_tokenize(text)
    # Keep only alphabetic tokens and remove stopwords
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    return tokens

# Tokenize all cleaned documents
tokenized_documents = {}
for filename, cleaned_text in cleaned_documents.items():
    tokenized_documents[filename] = tokenize_text(cleaned_text)
    print(f'  🔤 Tokenized: {filename} → {len(tokenized_documents[filename])} tokens')

print('\n✅ Tokenization complete!')

  🔤 Tokenized: 01_GenAI_Systems_Syllabus.docx → 228 tokens
  🔤 Tokenized: 06_Academic_Calendar_and_Policies.pdf → 347 tokens
  🔤 Tokenized: 07_IT_Ethics_Syllabus.pdf → 344 tokens
  🔤 Tokenized: 02_Business_Analytics_Syllabus.docx → 192 tokens
  🔤 Tokenized: 03_Information_Management_Syllabus.pdf → 257 tokens
  🔤 Tokenized: 04_Systems_Analysis_Design_Syllabus.pdf → 279 tokens
  🔤 Tokenized: 08_Software_Engineering_Syllabus.pdf → 334 tokens
  🔤 Tokenized: 05_Thesis_Writing_Syllabus.pdf → 298 tokens

✅ Tokenization complete!


## Step 8 — Tokenization Before/After Example

In [ ]:
sample_tokens = tokenized_documents[sample_filename]

print('=' * 60)
print('BEFORE TOKENIZATION (cleaned text excerpt):')
print('=' * 60)
print(sample_cleaned[:300])

print('\n' + '=' * 60)
print('AFTER TOKENIZATION (first 50 tokens):')
print('=' * 60)
print(sample_tokens[:50])
print(f'\n📊 Total unique tokens in this document: {len(set(sample_tokens))}')

BEFORE TOKENIZATION (cleaned text excerpt):
course syllabus
is professional elective 4 generative ai systems
instructor: jessie a. melendres 1st semester, a.y. 20262027 units: 3 schedule: tth 1:002:30 pm
1. course description
this course introduces students to the principles and applications of generative ai systems. topics include transforme

AFTER TOKENIZATION (first 50 tokens):
['course', 'syllabus', 'professional', 'elective', 'generative', 'ai', 'systems', 'instructor', 'jessie', 'melendres', 'semester', 'units', 'schedule', 'tth', 'pm', 'course', 'description', 'course', 'introduces', 'students', 'principles', 'applications', 'generative', 'ai', 'systems', 'topics', 'include', 'transformer', 'architecture', 'large', 'language', 'models', 'llms', 'prompt', 'engineering', 'generation', 'rag', 'vector', 'databases', 'techniques', 'lora', 'qlora', 'llmops', 'deployment', 'practices', 'students', 'design', 'build', 'working', 'generative']

📊 Total unique tokens in this document: 171


## Step 9 — Summary of All Documents

In [ ]:
print('=' * 60)
print('📋 PREPROCESSING SUMMARY — ALL DOCUMENTS')
print('=' * 60)
print(f'{"File":<40} {"Raw Chars":>10} {"Clean Chars":>12} {"Tokens":>8}')
print('-' * 72)

total_tokens = 0
for filename in raw_documents:
    raw_len   = len(raw_documents[filename])
    clean_len = len(cleaned_documents[filename])
    tok_count = len(tokenized_documents[filename])
    total_tokens += tok_count
    print(f'{filename:<40} {raw_len:>10} {clean_len:>12} {tok_count:>8}')

print('-' * 72)
print(f'{"TOTAL":<40} {"": >10} {"": >12} {total_tokens:>8}')
print(f'\n✅ Pipeline complete! {len(raw_documents)} documents preprocessed.')
print(f'📦 Total tokens across all syllabi: {total_tokens:,}')

📋 PREPROCESSING SUMMARY — ALL DOCUMENTS
File                                      Raw Chars  Clean Chars   Tokens
------------------------------------------------------------------------
01_GenAI_Systems_Syllabus.docx                 2656         2562      228
06_Academic_Calendar_and_Policies.pdf          3825         3791      347
07_IT_Ethics_Syllabus.pdf                      3583         3566      344
02_Business_Analytics_Syllabus.docx            2071         2025      192
03_Information_Management_Syllabus.pdf         2710         2686      257
04_Systems_Analysis_Design_Syllabus.pdf        2735         2713      279
08_Software_Engineering_Syllabus.pdf           3425         3348      334
05_Thesis_Writing_Syllabus.pdf                 3188         3174      298
------------------------------------------------------------------------
TOTAL                                                                2279

✅ Pipeline complete! 8 documents preprocessed.
📦 Total tokens across all 

## Step 10 — Save Cleaned Text Files
Save each cleaned document as a `.txt` file for use in the next steps (embedding generation).

In [ ]:
import os

os.makedirs('syllabi_cleaned', exist_ok=True)

for filename, cleaned_text in cleaned_documents.items():
    # Save as .txt with same base name
    base_name = os.path.splitext(filename)[0]
    output_path = f'syllabi_cleaned/{base_name}.txt'
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(cleaned_text)
    print(f'  💾 Saved: {output_path}')

print('\n✅ All cleaned files saved to /syllabi_cleaned/')
print('📌 These files will be used in the next notebook (Embedding Generation).')

  💾 Saved: syllabi_cleaned/01_GenAI_Systems_Syllabus.txt
  💾 Saved: syllabi_cleaned/06_Academic_Calendar_and_Policies.txt
  💾 Saved: syllabi_cleaned/07_IT_Ethics_Syllabus.txt
  💾 Saved: syllabi_cleaned/02_Business_Analytics_Syllabus.txt
  💾 Saved: syllabi_cleaned/03_Information_Management_Syllabus.txt
  💾 Saved: syllabi_cleaned/04_Systems_Analysis_Design_Syllabus.txt
  💾 Saved: syllabi_cleaned/08_Software_Engineering_Syllabus.txt
  💾 Saved: syllabi_cleaned/05_Thesis_Writing_Syllabus.txt

✅ All cleaned files saved to /syllabi_cleaned/
📌 These files will be used in the next notebook (Embedding Generation).


## Step 11 — Download Cleaned Text Files

Run this cell to compress all cleaned `.txt` files into a zip archive and download them to your local machine.

In [28]:
import shutil
from google.colab import files

output_zip_filename = 'syllabi_cleaned.zip'
shutil.make_archive(output_zip_filename.replace('.zip', ''), 'zip', 'syllabi_cleaned')

print(f'Compressing files into {output_zip_filename}...')
files.download(output_zip_filename)

print(f'\n✅ Download initiated for {output_zip_filename}')

Compressing files into syllabi_cleaned.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download initiated for syllabi_cleaned.zip
